Analysing the drift 
drift_patterns_over_time=[[1, 2],
                         [1, 2],
                         [2, 1]],

In [1]:
from __future__ import annotations

from typing import Dict, List, Sequence, Tuple, Optional, Any
import numpy as np
import pandas as pd

from log_utils.logging import read_logs  # same import as in your notebook

In [2]:
from log_utils.logging import read_logs
import numpy as np

### 1) Reading logs (same file layout as your notebook)

In [3]:
def load_drift_specs(drift_specs_pkl_path: str) -> Dict[str, Any]:
    """
    Loads drift_specs_log.pkl with keys like:
      - drift_clustered_client_indices
      - drift_step_rounds
    """
    return read_logs(drift_specs_pkl_path)

In [4]:
def load_client_log(client_log_pkl_path: str):
    """
    Loads client_log.pkl.
    Expected format (as in your notebook):
        client_log[t][client_id] = (loss, acc)
    """
    return read_logs(client_log_pkl_path)

In [5]:
def load_algo_logs_from_root(
    root: str,
    dataset: str,
    algo_to_subdir: Dict[str, str],
    *,
    drift_specs_from_algo: str = "FedAvg",
) -> Tuple[Dict[str, Any], Dict[str, Any]]:
    """
    Recreates what you do manually, but packaged:
      root/dataset/saved_logs_<subdir>/client_log.pkl
      root/dataset/saved_logs_<subdir>/drift_specs_log.pkl  (for one algo, usually FedAvg)

    Example (matches your notebook paths):
      root = "D://...//logs//swap"
      dataset = "CIFAR-10"
      algo_to_subdir = {"FedAvg":"fedavg", "FedEx":"fedex", "Oracle":"oracle"}
    """
    # drift specs
    specs_path = f"{root}//{dataset}//saved_logs_{algo_to_subdir[drift_specs_from_algo]}//drift_specs_log.pkl"
    drift_specs = load_drift_specs(specs_path)

    # per-algorithm client logs
    algo_logs = {}
    for algo_name, subdir in algo_to_subdir.items():
        p = f"{root}//{dataset}//saved_logs_{subdir}//client_log.pkl"
        algo_logs[algo_name] = load_client_log(p)

    return drift_specs, algo_logs

### 2) Cluster handling (reuses your drift_pattern logic)

In [6]:
def reorder_clusters_by_pattern(
    drift_clustered_client_indices: Sequence[Sequence[Sequence[int]]],
    drift_pattern: Sequence[Sequence[int]],
    keep_unmentioned: str = "append",  # "append" | "drop"
) -> List[List[List[int]]]:
    """
    Same idea as your notebook function.

    drift_pattern is 1-based indices per step, e.g.:
      scenario B: [[1,2], [1,2]]
      scenario C: [[1,2], [1,2], [2,1]]

    Returns a list of steps, each step is a list of clusters (lists of client ids).
    """
    if len(drift_clustered_client_indices) != len(drift_pattern):
        raise ValueError(
            f"len(drift_clustered_client_indices)={len(drift_clustered_client_indices)} "
            f"!= len(drift_pattern)={len(drift_pattern)}"
        )

    reordered: List[List[List[int]]] = []

    for step_clusters, order in zip(drift_clustered_client_indices, drift_pattern):
        k = len(step_clusters)

        for idx in order:
            if not (1 <= idx <= k):
                raise ValueError(f"Invalid cluster index {idx} for step with {k} clusters (valid: 1..{k}).")

        reordered_step = [list(step_clusters[i - 1]) for i in order]  # allow duplicates

        if keep_unmentioned == "append":
            mentioned = set(order)
            for j in range(1, k + 1):
                if j not in mentioned:
                    reordered_step.append(list(step_clusters[j - 1]))
        elif keep_unmentioned == "drop":
            pass
        else:
            raise ValueError("keep_unmentioned must be 'append' or 'drop'.")

        reordered.append(reordered_step)

    return reordered

In [7]:
def append_no_drift_cluster(
    clusters_per_step: Sequence[Sequence[Sequence[int]]],
    no_drift_clients: Sequence[int],
) -> List[List[List[int]]]:
    """
    Matches your notebook step:
      reordered_clusters_complete = [step_clusters + [extra_cluster] for step_clusters in reordered_clusters]
    """
    extra = list(no_drift_clients)
    return [ [list(c) for c in step] + [extra] for step in clusters_per_step ]


In [8]:
def ensure_final_segment_clusters(
    clusters_per_step: Sequence[Sequence[Sequence[int]]],
) -> List[List[List[int]]]:
    """
    Your drift steps define segments like [t0,t1), [t1,t2), ... and also [t_last, T).
    Often you have clusters for each *change step*, but you also need a definition for the last segment.
    The notebook commonly reuses the last definition for the final segment.
    """
    cps = [ [list(c) for c in step] for step in clusters_per_step ]
    if len(cps) == 0:
        return cps
    cps.append([list(c) for c in cps[-1]])  # reuse last
    return cps

### 3) Accuracy tables (rounds x algorithms) for each drift variant

In [9]:
def _make_segments(drift_steps, T):
    """
    drift_steps = [40, 65]
    => segments:
       [(0,40), (40,65), (65,T)]
    """
    ds = list(drift_steps)
    bounds = [0] + ds + [T]
    return list(zip(bounds[:-1], bounds[1:]))

In [10]:
def mean_acc_per_round_for_clients(client_log, clients: Sequence[int], start_t: int, end_t: int) -> Dict[int, float]:
    """
    Reuses your exact log format:
        client_log[t][c] = (loss, acc)
    """
    if len(clients) == 0:
        return {t: np.nan for t in range(start_t, end_t)}

    out = {}
    for t in range(start_t, end_t):
        accs = [client_log[t][c][1] for c in clients]
        out[t] = float(np.mean(accs))
    return out

In [11]:
def build_accuracy_tables(
    algo_logs: Dict[str, Any],
    drift_steps: Sequence[int],
    clusters_per_segment: Sequence[Sequence[Sequence[int]]],
    variant_names: Sequence[str],
    *,
    start_round: Optional[int] = None,
    end_round: Optional[int] = None,
) -> Dict[str, pd.DataFrame]:
    """
    Returns:
      tables[variant] = DataFrame
        index   = round
        columns = algorithm names
        values  = mean accuracy for that variant’s clients at each round

    IMPORTANT:
      clusters_per_segment must align with segments formed from drift_steps and T:
        segments = [(t0,t1), (t1,t2), ..., (t_last,T)]
      so len(clusters_per_segment) must equal len(segments).
    """
    if not algo_logs:
        raise ValueError("algo_logs is empty.")

    first_log = next(iter(algo_logs.values()))
    T = len(first_log)
    segments = _make_segments(drift_steps, T)

    if len(clusters_per_segment) != len(segments):
        raise ValueError(
            f"clusters_per_segment length ({len(clusters_per_segment)}) != number of segments ({len(segments)}).\n"
            f"Tip: call ensure_final_segment_clusters(...) if you only have step-wise clusters."
        )

    if len(clusters_per_segment[0]) != len(variant_names):
        raise ValueError(
            f"variant_names length ({len(variant_names)}) != number of clusters per segment ({len(clusters_per_segment[0])})."
        )

    r0 = drift_steps[0] if start_round is None else int(start_round)
    r1 = T if end_round is None else int(end_round)
    r0 = max(0, r0)
    r1 = min(T, r1)

    # tables[variant][t][algo] = acc
    tables: Dict[str, Dict[int, Dict[str, float]]] = {vn: {} for vn in variant_names}

    for (seg_start, seg_end), seg_clusters in zip(segments, clusters_per_segment):
        s = max(seg_start, r0)
        e = min(seg_end, r1)
        if s >= e:
            continue

        for cluster_idx, variant in enumerate(variant_names):
            clients = seg_clusters[cluster_idx]
            for algo_name, log in algo_logs.items():
                acc_by_round = mean_acc_per_round_for_clients(log, clients, s, e)
                for t, acc in acc_by_round.items():
                    tables[variant].setdefault(t, {})
                    tables[variant][t][algo_name] = acc

    out: Dict[str, pd.DataFrame] = {}
    for variant, by_round in tables.items():
        df = pd.DataFrame.from_dict(by_round, orient="index").sort_index()
        df.index.name = "round"
        out[variant] = df

    return out

### Fairness specific

In [12]:
from __future__ import annotations

from typing import Any, Dict, List, Optional, Sequence, Tuple, Union, Callable
import pandas as pd
import numpy as np

AggFn = Union[str, Callable[[pd.Series], float]]

# ----------------------------
# 1) Brand-new end-to-end loader
# ----------------------------
def load_logs_and_build_variant_tables_for_aeq(
    *,
    root: str,
    dataset: str,
    algo_to_subdir: Dict[str, str],
    drift_pattern: Sequence[Sequence[int]],
    no_drift_clients: Sequence[int],
    variant_names: Sequence[str] = ("drift_1", "drift_2", "no_drift"),
    drift_specs_from_algo: str = "FedAvg",
    start_round: int = 0,
    end_round: Optional[int] = None,
) -> Tuple[Dict[str, Any], Dict[str, Any], List[int], List[List[List[int]]], Dict[str, pd.DataFrame]]:
    """
    End-to-end pipeline (aligned with your notebook):
      - loads drift_specs + algo_logs (via your load_algo_logs_from_root)
      - extracts drift_steps + drift_clustered_client_indices (same keys as notebook)
      - reorders clusters by drift_pattern (your reorder_clusters_by_pattern)
      - appends 'no_drift' cluster (your append_no_drift_cluster)
      - ensures final segment clusters exist (your ensure_final_segment_clusters)
      - prepends a pre-drift cluster definition with all clients (same as notebook)
      - builds per-variant accuracy tables (your build_accuracy_tables)

    Returns:
      drift_specs, algo_logs, drift_steps, clusters_per_segment, tables_by_variant
    """
    # --- Read logs (your function) ---
    drift_specs, algo_logs = load_algo_logs_from_root(
        root=root,
        dataset=dataset,
        algo_to_subdir=algo_to_subdir,
        drift_specs_from_algo=drift_specs_from_algo,
    )

    drift_steps = list(drift_specs["drift_step_rounds"])
    drift_clustered_client_indices = drift_specs["drift_clustered_client_indices"]

    # --- Reorder clusters according to the scenario pattern (your function) ---
    reordered = reorder_clusters_by_pattern(
        drift_clustered_client_indices=drift_clustered_client_indices,
        drift_pattern=drift_pattern,
        keep_unmentioned="append",
    )

    # --- Add explicit no-drift cluster (your function) ---
    reordered_complete = append_no_drift_cluster(reordered, no_drift_clients)

    # --- Add final segment definition (your function) ---
    clusters_per_segment = ensure_final_segment_clusters(reordered_complete)

    # --- Prepend pre-drift segment as "all clients" (same as your cell 14) ---
    num_clients = len(next(iter(algo_logs.values()))[0])
    all_clients = list(range(num_clients))

    pre_drift_clusters = [all_clients for _ in variant_names]  # drift_1, drift_2, no_drift (artificial pre-drift)
    clusters_per_segment = [pre_drift_clusters] + clusters_per_segment

    # --- Build per-variant accuracy tables (your function) ---
    tables_by_variant = build_accuracy_tables(
        algo_logs=algo_logs,
        drift_steps=drift_steps,
        clusters_per_segment=clusters_per_segment,
        variant_names=list(variant_names),
        start_round=start_round,
        end_round=end_round,
    )

    return drift_specs, algo_logs, drift_steps, clusters_per_segment, tables_by_variant


# ----------------------------
# 2) AEQ computation (drift-onset summary)
# ----------------------------
def _rolling_aggregate(series: pd.Series, window: int, agg: AggFn = "mean") -> pd.Series:
    """
    Trailing rolling aggregation.
    window=1 -> identity.
    """
    if window <= 1:
        return series.copy()

    roll = series.rolling(window=window, min_periods=window)
    if isinstance(agg, str):
        a = agg.lower()
        if a == "mean":
            return roll.mean()
        if a == "median":
            return roll.median()
        if a == "min":
            return roll.min()
        if a == "max":
            return roll.max()
        raise ValueError(f"Unknown agg='{agg}'. Use mean/median/min/max or callable.")
    else:
        return roll.apply(lambda x: agg(pd.Series(x)), raw=False)


def _aggregate_horizon(values: pd.Series, agg: str = "mean") -> float:
    """
    Aggregate AEQ over a post-onset horizon. Ignores NaNs by default.
    """
    a = agg.lower()
    if a == "mean":
        return float(values.mean(skipna=True))
    if a == "median":
        return float(values.median(skipna=True))
    if a == "min":
        return float(values.min(skipna=True))
    if a == "max":
        return float(values.max(skipna=True))
    if a == "last":
        vv = values.dropna()
        return float(vv.iloc[-1]) if len(vv) else float("nan")
    raise ValueError("horizon_agg must be one of: mean/median/min/max/last")


def build_aeq_at_drift_onsets_table(
    tables_by_variant: Dict[str, pd.DataFrame],
    *,
    drift_steps: Sequence[int],
    algorithms: Sequence[str] = ("FedAvg", "FedEx", "Oracle"),
    group_num: str = "drift_1",
    group_den: str = "drift_2",
    smooth_window: int = 1,
    smooth_agg: AggFn = "mean",
    horizon_rounds: int = 0,
    horizon_agg: str = "mean",
    eps: float = 1e-12,
    decimals: int = 4,
) -> pd.DataFrame:
    """
    Computes AEQ at each drift onset:
      AEQ(t) = Acc_num(t) / Acc_den(t)
    where Acc_* can be smoothed by a trailing window over rounds.

    If horizon_rounds > 0:
      We aggregate AEQ over [t, t+horizon_rounds] using horizon_agg.

    Output DF:
      index: onset round (int)
      columns: algorithms
      values: AEQ summary per onset
    """
    if group_num not in tables_by_variant or group_den not in tables_by_variant:
        raise KeyError(f"Unknown group(s). Available groups: {list(tables_by_variant.keys())}")

    df_num = tables_by_variant[group_num]
    df_den = tables_by_variant[group_den]

    # align index (rounds)
    rounds = df_num.index.intersection(df_den.index)

    out = pd.DataFrame(index=list(drift_steps), columns=list(algorithms), dtype=float)
    out.index.name = "onset_round"

    for algo in algorithms:
        if algo not in df_num.columns or algo not in df_den.columns:
            raise KeyError(
                f"Algorithm '{algo}' missing from group tables. "
                f"num cols={list(df_num.columns)}, den cols={list(df_den.columns)}"
            )

        s_num = _rolling_aggregate(df_num.loc[rounds, algo].astype(float), smooth_window, smooth_agg)
        s_den = _rolling_aggregate(df_den.loc[rounds, algo].astype(float), smooth_window, smooth_agg)
        aeq = s_num / (s_den + eps)

        for t in drift_steps:
            if t not in aeq.index:
                out.loc[t, algo] = np.nan
                continue

            if horizon_rounds <= 0:
                out.loc[t, algo] = float(aeq.loc[t])
            else:
                horizon_idx = [r for r in range(int(t), int(t) + int(horizon_rounds) + 1) if r in aeq.index]
                vals = aeq.loc[horizon_idx]
                out.loc[t, algo] = _aggregate_horizon(vals, horizon_agg)

    return out.round(decimals)


# ----------------------------
# 3) IEEE LaTeX exporter (brand-new, but consistent with your style)
# ----------------------------
def latex_ieee_table_aeq_onsets(
    aeq_onsets: pd.DataFrame,
    *,
    scenario_label: str,
    ratio_label: Optional[str] = None,  # if None -> derived from groups in caption or left blank
    onset_formatter: Optional[Callable[[int], str]] = None,  # use your make_onset_formatter(...)
    caption: str = "Accuracy Equality (AEQ) around drift onsets.",
    label: str = "tab:aeq_onsets",
    tabcolsep_pt: int = 3,
    algorithms: Sequence[str] = ("FedAvg", "FedEx", "Oracle"),
    value_decimals: int = 2,
) -> str:
    """
    Produces an IEEE-style LaTeX table:

      Scenario | Ratio | Onset | FedAvg | FedEx | Oracle

    onset_formatter:
      - If you want constant percentages (no conversion), pass:
          onset_formatter = make_onset_formatter(mode="percent", mapping={20:"40\\%", 33:"65\\%"})
      - Or pass None to print the raw onset round integer.
    """
    def fmt_onset(t: int) -> str:
        if onset_formatter is None:
            return str(int(t))
        return onset_formatter(int(t))

    def fmt_val(x: Any) -> str:
        if x is None or (isinstance(x, float) and np.isnan(x)):
            return "--"
        return f"{float(x):.{value_decimals}f}"

    # Header
    cols = ["Scenario", "Ratio", "Onset"] + list(algorithms)
    colspec = "ccc" + ("c" * len(algorithms))

    lines = []
    lines.append("\\begin{table}[t]")
    lines.append(f"  \\caption{{{caption}}}")
    lines.append(f"  \\label{{{label}}}")
    lines.append("  \\begin{center}")
    lines.append("    \\begin{small}")
    lines.append("      \\begin{sc}")
    lines.append(f"        \\setlength{{\\tabcolsep}}{{{tabcolsep_pt}pt}}")
    lines.append(f"        \\begin{{tabular}}{{{colspec}}}")
    lines.append("          \\toprule")
    lines.append("          " + " & ".join(cols) + " \\\\")
    lines.append("          \\midrule")

    # Body
    ratio_str = ratio_label if ratio_label is not None else ""

    for onset, row in aeq_onsets.iterrows():
        vals = [fmt_val(row.get(algo, np.nan)) for algo in algorithms]
        lines.append(
            "          "
            + " & ".join([scenario_label, ratio_str, fmt_onset(int(onset))] + vals)
            + " \\\\"
        )

    # Footer
    lines.append("          \\bottomrule")
    lines.append("        \\end{tabular}")
    lines.append("      \\end{sc}")
    lines.append("    \\end{small}")
    lines.append("  \\end{center}")
    lines.append("\\end{table}")

    return "\n".join(lines)


### LAtex tables

In [13]:
def filter_true_drift_onsets(drift_steps: list[int], *, drop_last: bool = True) -> list[int]:
    """
    Many logs include the final training round as the last 'drift step' to close the last segment.
    For onset-based metrics/tables, we typically drop that last marker.
    """
    if not drift_steps:
        return []
    return drift_steps[:-1] if drop_last else drift_steps


In [14]:
def make_onset_formatter(
    *,
    mode: str = "round",
    mapping: dict | None = None,
    fallback_to_round: bool = True,
) -> callable:
    mode = mode.lower()

    if mode == "round":
        return lambda t: str(int(t))

    if mode == "percent":
        if mapping is None:
            raise ValueError("mode='percent' requires a mapping dict")

        def _fmt(t: int) -> str:
            t = int(t)
            if t in mapping:
                return mapping[t]
            return str(t) if fallback_to_round else ""

        return _fmt

    raise ValueError(f"Unknown mode='{mode}'")


In [15]:
def make_percent_onset_formatter_from_labels(
    drift_onsets: list[int],
    labels: list[str],
) -> callable:
    if len(drift_onsets) != len(labels):
        raise ValueError(f"Need same length: drift_onsets={len(drift_onsets)} labels={len(labels)}")
    mapping = {int(r): lab for r, lab in zip(drift_onsets, labels)}
    return make_onset_formatter(mode="percent", mapping=mapping)


### 4) Usage

Step 1 — load logs + drift specs

In [52]:
ROOT = "D://publications_related//2026//ICDCS//logs//swap"
DATASET = "CIFAR-10"

algo_to_subdir = {"FedAvg":"fedavg", "FedEx":"fedex", "Oracle":"oracle"}

# Example drift pattern (you used patterns like this in the notebook)
# drift_pattern is 1-based indices per step
# drift_pattern=[[1], [1, 2]]  # scenario A
drift_pattern=[[1, 2], [1, 2], [2, 1]] # scenario C
# drift_pattern = [[1,2], [1,2]]   # Scenario B

no_drift_clients = [8, 9]

drift_specs, algo_logs, drift_steps, clusters_per_segment, tables_by_variant = \
    load_logs_and_build_variant_tables_for_aeq(
        root=ROOT,
        dataset=DATASET,
        algo_to_subdir=algo_to_subdir,
        drift_pattern=drift_pattern,
        no_drift_clients=no_drift_clients,
        start_round=0,
    )

In [53]:
drift_onsets = filter_true_drift_onsets(drift_steps, drop_last=True)

aeq_onsets = build_aeq_at_drift_onsets_table(
    tables_by_variant=tables_by_variant,
    drift_steps=drift_onsets,                     # ✅ use filtered onsets
    algorithms=["FedAvg", "FedEx", "Oracle"],
    group_num="drift_1",
    group_den="drift_2",
    smooth_window=2,
    horizon_rounds=20,
    horizon_agg="mean",
    decimals=4,
)

# 3) formatter: choose labels matching how many onsets you have
# scenario_percentages = ["40\\%","65\\%"]  # Scenaio A/B
scenario_percentages = ["40\\%","65\\%","70\\%"]   # scenaio C
onset_formatter = make_percent_onset_formatter_from_labels(
    drift_onsets,
    scenario_percentages,   # adjust to your scenario
)

# onset_formatter = make_onset_formatter(
#     mode="percent",
#     mapping={drift_onsets[0]: "40\\%", drift_onsets[1]: "65\\%"}  # ✅ no 50 anymore
# )

latex = latex_ieee_table_aeq_onsets(
    aeq_onsets,
    scenario_label="C",
    ratio_label="drift_1/drift_2",
    onset_formatter=onset_formatter,
    caption="Accuracy Equality (AEQ) at drift onsets (mean over 5 rounds post-onset).",
    label="tab:aeq_onsets_C",
    tabcolsep_pt=3,
    algorithms=["FedAvg", "FedEx", "Oracle"],
    value_decimals=2,
)

print(aeq_onsets)
print(latex)



             FedAvg   FedEx  Oracle
onset_round                        
40           0.9934  0.9826  0.9605
65           1.0353  0.9287  1.0058
70           1.0140  0.9345  1.0317
\begin{table}[t]
  \caption{Accuracy Equality (AEQ) at drift onsets (mean over 5 rounds post-onset).}
  \label{tab:aeq_onsets_C}
  \begin{center}
    \begin{small}
      \begin{sc}
        \setlength{\tabcolsep}{3pt}
        \begin{tabular}{cccccc}
          \toprule
          Scenario & Ratio & Onset & FedAvg & FedEx & Oracle \\
          \midrule
          C & drift_1/drift_2 & 40\% & 0.99 & 0.98 & 0.96 \\
          C & drift_1/drift_2 & 65\% & 1.04 & 0.93 & 1.01 \\
          C & drift_1/drift_2 & 70\% & 1.01 & 0.93 & 1.03 \\
          \bottomrule
        \end{tabular}
      \end{sc}
    \end{small}
  \end{center}
\end{table}


In [54]:
aeq_onsets = build_aeq_at_drift_onsets_table(
    tables_by_variant=tables_by_variant,
    drift_steps=drift_onsets,                     # ✅ use filtered onsets
    algorithms=["FedAvg", "FedEx", "Oracle"],
    group_num="drift_2",
    group_den="no_drift",
    smooth_window=2,
    horizon_rounds=20,
    horizon_agg="mean",
    decimals=4,
)

# 3) formatter: choose labels matching how many onsets you have
# scenario_percentages = ["40\\%","65\\%"]  #Scenaio A/B
scenario_percentages = ["40\\%","65\\%","70\\%"]   # scenaio C
onset_formatter = make_percent_onset_formatter_from_labels(
    drift_onsets,
    scenario_percentages,   # adjust to your scenario
)

latex = latex_ieee_table_aeq_onsets(
    aeq_onsets,
    scenario_label="C",
    ratio_label="drift_2/no_drift",
    onset_formatter=onset_formatter,
    caption="Accuracy Equality (AEQ) at drift onsets (mean over 5 rounds post-onset).",
    label="tab:aeq_onsets_C",
    tabcolsep_pt=3,
    algorithms=["FedAvg", "FedEx", "Oracle"],
    value_decimals=2,
)

print(latex)
print(aeq_onsets)


\begin{table}[t]
  \caption{Accuracy Equality (AEQ) at drift onsets (mean over 5 rounds post-onset).}
  \label{tab:aeq_onsets_C}
  \begin{center}
    \begin{small}
      \begin{sc}
        \setlength{\tabcolsep}{3pt}
        \begin{tabular}{cccccc}
          \toprule
          Scenario & Ratio & Onset & FedAvg & FedEx & Oracle \\
          \midrule
          C & drift_2/no_drift & 40\% & 0.84 & 0.92 & 1.02 \\
          C & drift_2/no_drift & 65\% & 0.87 & 0.92 & 0.98 \\
          C & drift_2/no_drift & 70\% & 0.87 & 0.94 & 0.97 \\
          \bottomrule
        \end{tabular}
      \end{sc}
    \end{small}
  \end{center}
\end{table}
             FedAvg   FedEx  Oracle
onset_round                        
40           0.8365  0.9249  1.0239
65           0.8703  0.9179  0.9751
70           0.8682  0.9375  0.9743


In [55]:
aeq_onsets = build_aeq_at_drift_onsets_table(
    tables_by_variant=tables_by_variant,
    drift_steps=drift_onsets,                     # ✅ use filtered onsets
    algorithms=["FedAvg", "FedEx", "Oracle"],
    group_num="no_drift",
    group_den="drift_1",
    smooth_window=2,
    horizon_rounds=20,
    horizon_agg="mean",
    decimals=4,
)


# 3) formatter: choose labels matching how many onsets you have
# scenario_percentages = ["40\\%","65\\%"]  # Scenaio A/B
scenario_percentages = ["40\\%","65\\%","70\\%"]   # scenaio C
onset_formatter = make_percent_onset_formatter_from_labels(
    drift_onsets,
    scenario_percentages,   # adjust to your scenario
)

latex = latex_ieee_table_aeq_onsets(
    aeq_onsets,
    scenario_label="C",
    ratio_label="drift_1/no_drift",
    onset_formatter=onset_formatter,
    caption="Accuracy Equality (AEQ) at drift onsets (mean over 5 rounds post-onset).",
    label="tab:aeq_onsets_C",
    tabcolsep_pt=3,
    algorithms=["FedAvg", "FedEx", "Oracle"],
    value_decimals=2,
)

print(latex)
print(aeq_onsets)

\begin{table}[t]
  \caption{Accuracy Equality (AEQ) at drift onsets (mean over 5 rounds post-onset).}
  \label{tab:aeq_onsets_C}
  \begin{center}
    \begin{small}
      \begin{sc}
        \setlength{\tabcolsep}{3pt}
        \begin{tabular}{cccccc}
          \toprule
          Scenario & Ratio & Onset & FedAvg & FedEx & Oracle \\
          \midrule
          C & drift_1/no_drift & 40\% & 1.20 & 1.10 & 1.02 \\
          C & drift_1/no_drift & 65\% & 1.11 & 1.18 & 1.02 \\
          C & drift_1/no_drift & 70\% & 1.14 & 1.15 & 1.00 \\
          \bottomrule
        \end{tabular}
      \end{sc}
    \end{small}
  \end{center}
\end{table}
             FedAvg   FedEx  Oracle
onset_round                        
40           1.2046  1.1046  1.0204
65           1.1125  1.1790  1.0241
70           1.1363  1.1469  0.9978


Step 2 — scenario pattern + cluster prep (same as your cells)

In [507]:
# drift_pattern = [[1, 2], [1, 2]]# Example: Scenario B
drift_pattern=[[1], [1, 2]]  # scenario A
# drift_pattern=[[1, 2], [1, 2], [2, 1]] # scenario C

reordered = reorder_clusters_by_pattern(drift_clustered_client_indices, drift_pattern)

In [508]:
 # scenario A
reordered[0] = reordered[0][0]+reordered[0][1]  # merging the groups with drift ID~1 to be 1 list
reordered[0] = [reordered[0],[]]

In [509]:
# In your notebook you used extra_cluster = [8,9] for no-drift
no_drift_clients = [8, 9]
reordered_complete = append_no_drift_cluster(reordered, no_drift_clients)

# Make sure we also have a definition for the final segment [last_drift, T)
clusters_per_segment = ensure_final_segment_clusters(reordered_complete)

num_clients = len(next(iter(algo_logs.values()))[0])

all_clients = list(range(num_clients))

pre_drift_clusters = [
    all_clients,  # drift_1 (artificial)
    all_clients,  # drift_2 (artificial)
    all_clients,  # no_drift
]

clusters_per_segment = [pre_drift_clusters] + clusters_per_segment

variant_names = ["drift_1", "drift_2", "no_drift"]